# Create Initial Conditions for Standalone CICE6

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import cftime
import datetime
import cmocean as cm
import cartopy.crs as ccrs
import cartopy.feature as cft
import sys, os, warnings
import dask
from dask.distributed import Client
from datetime import timedelta
import glob
import os
from datatree import DataTree, map_over_subtree
import re
import socket
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean.cm as cmo
import matplotlib.lines as mlines
import cartopy.feature as cft
import matplotlib.colors as mcolors
import calendar

warnings.filterwarnings('ignore')

In [ ]:
# import os

# if os.path.islink(link_name):
#     os.remove(link_name)
#     print(f"Symlink removed: {link_name}")
# else:
#     print(f"No symlink found at: {link_name}")

In [ ]:
ic_file = "/g/data/vk83/configurations/inputs/access-om3/cice/initial_conditions/global.1deg/2023.07.28/iced.1900-01-01-10800.nc"
ic_dir = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-10/initial_conditions/2023.07.28"

sst_sss_file = "/g/data/vk83/experiments/inputs/access-om2/ice/initial_conditions/global.1deg/2020.05.30/monthly_sstsss.nc"
ocn_dir = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-10/ocean/2020.05.30"

# Create symlink for ic
os.makedirs(ic_dir, exist_ok=True)
link_name = os.path.join(ic_dir, os.path.basename(ic_file))
try:
    os.symlink(ic_file, link_name)
    print(f"Symlink created: {link_name} → {ic_file}")
except FileExistsError:
    print(f"Symlink already exists: {link_name}")


# Create symlink for ocean
os.makedirs(ocn_dir, exist_ok=True)
link_name = os.path.join(ocn_dir, os.path.basename(sst_sss_file))
try:
    os.symlink(sst_sss_file, link_name)
    print(f"Symlink created: {link_name} → {ic_file}")
except FileExistsError:
    print(f"Symlink already exists: {link_name}")


In [ ]:
ds_ic = xr.open_mfdataset(
        ic_file, 
        # chunks="auto",
        combine="by_coords", 
        decode_times=True,
        decode_timedelta=False,
    )
# ds["time"] = ds.time.to_pandas()
ds_ic

In [ ]:
ds_sst = xr.open_mfdataset(
        sst_sss_file, 
        # chunks="auto",
        combine="by_coords", 
        decode_times=True,
        decode_timedelta=False,
    )
# ds["time"] = ds.time.to_pandas()
# ds_sst
ds_sst = ds_sst.rename({
    "GRID_Y_T": "nj",
    "GRID_X_T": "ni",
    "TMONTH": "time"
})

ds_sst = ds_sst.rename_vars({
    "TEMP": "sst",
    "SALT": "sss"
})

ds_sst["sst"].attrs["long_name"] = "Sea Surface Temperature"
ds_sst["sss"].attrs["long_name"] = "Sea Surface Salinity"

ds_sst

In [ ]:
ds_ic['sst'] = ds_sst['sst'].isel(time=0)
ds_ic['sss'] = ds_sst['sss'].isel(time=0)

In [ ]:
ds_ic['sst'].plot()

In [ ]:
ds_ic['sss'].plot()

In [ ]:
ds_ic["frzmlt"] = ds_ic["sst"].copy()
ds_ic["frzmlt"].attrs["long_name"] = "Freeze/melt potential [W/m^2]"
ds_ic["frzmlt"].data[:] = 0.0
ds_ic["frzmlt"].plot()

In [ ]:
ds_ic.to_netcdf(os.path.join(ic_dir, "iced.1900-01-01-10800-sst.nc"))